# Rhea FinGraph — Temporal GNN Training on Kaggle T4 (Step 4)

Trains the **TeMP-TraG-style temporal heterogeneous GNN** (PyTorch Geometric) on the free Tesla T4 GPU (zero MacBook heat):

1. `graph_snapshots` — builds leakage-safe temporal snapshots (yearly buckets) from the parquet splits; node features come from **strictly-past** history only
2. `train_gnn` — trains the TemporalHeteroGNN (heterogeneous message passing + causal temporal transformer + edge MLP) and a HomogeneousGraphSAGE baseline for comparison

**Before running:** Session options → Accelerator → **GPU T4 x2** (falls back to CPU automatically if left off), and Input must include your `rhea-fingraph-ibm-splits` dataset (train / validation / test parquets).
**After running:** File → Save Version → **Save & Run All (Commit)**, wait for *Save complete*, then download `rhea_gnn_artifacts.zip` from the Output page.

In [ ]:
%pip install -q -U polars
%pip install -q torch-geometric
%pip install -q --force-reinstall --no-deps git+https://github.com/aditisahu1234/Rhea-FinGraph.git

In [ ]:
import glob
import os
import subprocess
import sys
from pathlib import Path

# locate the dataset parquets
by_name = {p.split("/")[-1]: p for p in glob.glob("/kaggle/input/**/*.parquet", recursive=True)}
print("Found:", sorted(by_name))
assert "train.parquet" in by_name and "validation.parquet" in by_name
assert "test.parquet" in by_name

# copy to a predictable layout (the package expects ./data/processed/ibm_full/)
target = Path("data/processed/ibm_full")
target.mkdir(parents=True, exist_ok=True)
for name in ("train.parquet", "validation.parquet", "test.parquet"):
    dest = target / name
    if not dest.exists():
        print(f"copying {by_name[name]} -> {dest} ...", flush=True)
        os.symlink(by_name[name], dest)

DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print(f"Device: {DEVICE}", flush=True)
if DEVICE == "cpu":
    print(
        "WARNING: no GPU detected -- Session options > Accelerator > GPU T4 x2",
        "and re-run.",
        flush=True,
    )


## 1) Build temporal snapshots (yearly buckets)

~29 snapshots from 1991→2020. Node features are history-only (no leakage).

In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "fingraph_sentinel.graph_snapshots",
        "--splits", "train", "val", "test",
        "--bucket-months", "12",
        "--out", "/kaggle/working/graph",
    ],
    check=True,
)

## 2) Train the temporal heterogeneous GNN (+ GraphSAGE baseline)

Chronological split: earliest 60% of snapshots = train, next 20% = validation, newest 20% = test (locked).

In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "fingraph_sentinel.train_gnn",
        "--data-dir", "/kaggle/working/graph",
        "--out", "/kaggle/working/gnn",
        "--device", DEVICE,
        "--epochs", "12",
        "--hidden", "64",
        "--layers", "2",
        "--heads", "4",
        "--with-sage",
    ],
    check=True,
)

## 3) Package artifacts

Download `rhea_gnn_artifacts.zip` from the Output page after the Commit finishes.

In [ ]:
import json

config_path = Path("/kaggle/working/gnn/gnn_config.json")
if config_path.exists():
    cfg = json.loads(config_path.read_text())
    print("== GNN RESULTS ==")
    print(f"  validation:  {cfg.get('metrics_validation')}")
    print(f"  test locked: {cfg.get('metrics_test_locked')}")
    print(f"  fit_seconds: {cfg.get('fit_seconds')}")

!cd /kaggle/working && zip -qr rhea_gnn_artifacts.zip gnn graph
!ls -lh /kaggle/working/rhea_gnn_artifacts.zip
print("\nDone. Download rhea_gnn_artifacts.zip from the Output page.")